# Proyecto Dashboard Stock

#### Contexto del Proyecto

In [74]:

#### ESTRUCTURA DEL PROYECTO DE INTEGRACIÓN (Módulos 2, 3 y 4):
# ---------------------------------------------------------
# Este archivo sirve como base lógica para migrar el sistema de fórmulas de Google Sheets 
# a un entorno de Python (Pandas/Scikit-Learn), cumpliendo con los objetivos del 
# Diplomado en IA para Ciencias de la Salud de la USM[cite: 1].

# Esta hoja de google sheet StockAntibióticos recibe datos de unos formularios de google forms, por lo que cada fila corresponde 
# a una respuesta de un formulario, y cada columna corresponde a una pregunta del formulario.

# Registra recepción de antibióticos para test de susceptibilidad de bacterias de la unidad de microbiología. 
# Los antibióticos, se reciben en 2 presentaciones: E-Test y sensidiscos. 
# Para conveniencia, las cajas de E-test se registran de a 1 (25 unidades aprox), mientras que en cada caja de sensidisco vienen 
# 5 tubos con 50 discos cada uno, por lo que se registra cada tubo como una unidad (50 discos) cada caja son 5 unidades de sensidiscos.

# 1. FUENTE DE DATOS: 
#   Archivo: 'Registro de antibióticos (respuestas)'[cite: 2]
#   Hoja: 'StockAntibióticos' (Rango A1:L193+)[cite: 2]

# 2. OBJETIVOS TÉCNICOS POR MÓDULO[cite: 1]:
#   - Módulo 2 (Python): Conexión vía API (gspread) y manipulación de estructuras de datos.
#   - Módulo 3 (Datos): Limpieza de fechas de apertura/vencimiento y cálculo de stock neto.
#   - Módulo 4 (ML): Predicción de quiebre de stock basado en consumo histórico.
#3. ÉTICA Y SEGURIDAD[cite: 1]:
#   - Aunque los datos son de insumos (antibióticos) y no de pacientes, se debe mantener 
#     trazabilidad total de quién registra (Columna: Responsable)[cite: 2].
#   - El código debe ser auditable para asegurar la disponibilidad de fármacos críticos.
#"""


#--- MAPEO DE COLUMNAS SEGÚN HOJA 'StockAntibióticos'[cite: 2] ---

#    'A': 'marca_temporal',
#    'B': 'tipo_registro',      # Apertura, Recepción, Evento[cite: 2]
#    'C': 'antibiotico',        # Ej: "Amikacina (Sensidisco)"[cite: 2]
#    'E': 'fecha_recepcion',
#    'G': 'fecha_apertura',
#    'K': 'fecha_vencimiento',  # Variable crítica para alertas[cite: 2]
#    'L': 'cantidad_eliminada'
#}

#-- LÓGICA DE EVENTOS ---
 #   Existen 3 tipos de eventos:
 #       Evento por quiebre de stock.
 #       Evento por control de calidad fallido
 #       Evento por vencimiento.
 #   Los 2 primeros funcionan solo como registro del evento.
#    El evento por vencimiento también indica cuántas cantidades fueron eliminadas por caducadas.
#    Los eventos NO INDICAN que se acabó un antibiótico, funcionan solo como marca temporal.


#--- LÓGICA DE CÁLCULO CORRECTA ---
#    La lógica para calcular la duración de cada unidad es calculando el tiempo entre 2 aperturas de un mismo antibiótico siempre y cuando
#    no haya un evento de ese mismo antibiótico entre 2 aperturas consecutivas.


#--- RECORDATORIO ÉTICO USM[cite: 1] ---
#1. Calidad del Dato: "No son personas, son antibióticos" (Ledger de corrección).
#   Un error en el script de stock podría dejar sin un tratamiento adecuado 
#2. Transparencia: El código debe documentar por qué se recomienda una compra.


# La Dimensión Técnica: Antibióticos como Herramientas de Diagnóstico
# En un laboratorio de microbiología, los antibióticos (en formato de sensidiscos o E-test) no son fármacos terapéuticos, 
# sino reactivos de diagnóstico crítico. Su función no es curar al paciente directamente, sino actuar como sensores biológicos 
# para determinar el perfil de susceptibilidad de un patógeno.  

# Impacto de la Ruptura de Stock: Si la hoja de StockAntibióticos muestra un quiebre de stock (stock = 0), 
# la consecuencia no es la falta de un tratamiento, sino la generación de un vacío de información diagnóstica.  

#Consecuencia Clínica: Sin el insumo para el testeo, el equipo médico queda "a ciegas", perdiendo la capacidad de
# realizar una terapia dirigida y viéndose obligado a mantener esquemas empíricos de amplio espectro, lo que aumenta 
# la presión selectiva y la resistencia bacteriana.  

# El Rol de la IA (Módulo 4): Tu proyecto de integración busca precisamente evitar este escenario mediante la predicción de demanda,
# asegurando que la capacidad diagnóstica del hospital nunca se vea interrumpida.

#### Importaciones y Conexión a Google Sheets

In [75]:
import gspread
import pandas as pd
import numpy as np
from google.oauth2.service_account import Credentials
from datetime import datetime, timedelta

scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
creds = Credentials.from_service_account_file("credentials.json", scopes=scopes)

gc = gspread.authorize(creds)

# Open the Google Sheet by name or URL
sheet = gc.open("Registro de antibióticos (respuestas)")

# Para tener las dos hojas de trabajo:
worksheet_atb = sheet.worksheet("StockAntibióticos")
worksheet_ins = sheet.worksheet("StockInsumos")


# Leer toda la data de worksheet_atb y worksheet_ins
data_atb = worksheet_atb.get_all_values()
data_ins = worksheet_ins.get_all_values()   

# para abrir el chat incorporado con gemini en vs code debo hacer click en el icono de la esquina inferior derecha que dice "Chat" y luego seleccionar "Gemini" en el menú desplegable.
#  Si no encuentras el icono de "Chat", puedes abrir la paleta de comandos con Ctrl+Shift+P (Cmd+Shift+P en Mac) y escribir "Chat: Open Chat" para abrirlo manualmente. Luego, selecciona "Gemini" como tu asistente de chat para comenzar a interactuar con él.


# Authorize gspread
gc = gspread.authorize(creds)


# Option 2: By URL
# sheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/YOUR_SHEET_ID/edit")

# Get the first worksheet
# worksheet = sheet.worksheet(0)

# Read all data
# data = worksheet.get_all_values()


# 2. Configurar el ancho total de la pantalla (en caracteres)
# Aumenta este valor (p.ej., 200 o 300) según el tamaño de tu monitor
pd.set_option('display.width', 2000)

# crear dataframe de pandas a partir de los datos obtenidos
df_atb = pd.DataFrame(data_atb[1:], columns=data_atb[0])
df_ins = pd.DataFrame(data_ins[1:], columns=data_ins[0])
# Display the data
print(df_atb)
print(df_ins)

# 2. Configurar el ancho total de la pantalla (en caracteres)
# Aumenta este valor (p.ej., 200 o 300) según el tamaño de tu monitor
pd.set_option('display.width', 2000)

print("ok")

          Marca temporal Tipo de registro                               Antibiótico Responsable (iniciales) Fecha de recepción Cantidad (En caso de sensidiscos, indicar cantidad de tubos) Fecha de apertura Fecha de cierre            Motivo de cierre Comentario (opcional) Fecha de vencimiento Cantidad eliminada
0      6/01/2026 8:02:24         Apertura                    Aztreonam (Sensidisco)                     SCS                                                                                        29/08/2025                                                                                                          
1      6/01/2026 8:02:39         Apertura                    Amikacina (Sensidisco)                     SCS                                                                                        29/08/2025                                                                                                          
2      6/01/2026 8:02:56         Apertura               Ciproflo

#### Exploración y Selección de Datos

In [76]:
dayfirst=True
# Exploración de los datos: 
# print(df.shape)
#print(df.columns)
#print(df.head())

#para comprobar el tipo de dato de cada columna, podemos usar el siguiente código:
# print(df.dtypes)

#vamos a mostrar el nombre de las columnas para tener una referencia clara de cómo se llaman:
# print(df.columns)

#vamos a renombrar las columnas para que tengan nombres más amigables y fáciles de usar para ambos dataframes, ya que ambos tienen las mismas columnas pero con nombres diferentes, 
# por lo que vamos a crear un mapeo de columnas para renombrarlas de manera consistente para ambos dataframes:
mapeo_columnas = {
    'Marca temporal': 'marca_temporal',
    'Tipo de registro': 'tipo_registro',
    'Antibiótico': 'antibiotico',
    'Insumo': 'insumo',
    'Fecha de recepción': 'fecha_recepcion',
    'Fecha de apertura': 'fecha_apertura',
    'Fecha de evento': 'fecha_evento',
    'Fecha de vencimiento': 'fecha_vencimiento',
    'Cantidad': 'cantidad_recibida',
    'Cantidad eliminada': 'cantidad_eliminada',
    'Responsable': 'responsable',
}

#renombramos las columnas de ambos dataframe usando el mapeo definido anteriormente:
df_atb = df_atb.rename(columns=mapeo_columnas)
df_ins = df_ins.rename(columns=mapeo_columnas)

# para comprobar que se renombraron correctamente las columnas, podemos imprimir las columnas nuevamente:
# print("columnas df_atb:", df_atb.columns)
# print("columnas df_ins:", df_ins.columns)

#Eliminaremos las columnas que no nos son útiles para el análisis, como
# timestamp, responsable, comentario, ya que no aportan información relevante para el cálculo de stock y predicción de quiebre.
columnas_a_eliminar = ['responsable', 'timestamp', 'comentario']
# esto da error porque
df_atb = df_atb.drop(columns=columnas_a_eliminar, errors='ignore')
df_ins = df_ins.drop(columns=columnas_a_eliminar, errors='ignore')
# para comprobar que se eliminaron las columnas correctamente, podemos imprimir las columnas nuevamente:
print("df_atb.head():", df_atb.head())
print("df_ins.tail():", df_ins.tail())

# para convertir las columnas a su formato correcto, podemos usar el siguiente código:

df_atb['cantidad_recibida'] = pd.to_numeric(df_atb['cantidad_recibida'], errors='coerce').astype('Int64')
df_atb['cantidad_eliminada'] = pd.to_numeric(df_atb['cantidad_eliminada'], errors='coerce').astype('Int64')

df_ins['cantidad_recibida'] = pd.to_numeric(df_ins['cantidad_recibida'], errors='coerce').astype('Int64')
df_ins['cantidad_eliminada'] = pd.to_numeric(df_ins['cantidad_eliminada'], errors='coerce').astype('Int64')

columnas_fecha = ['fecha_recepcion', 'fecha_apertura', 'fecha_evento', 'fecha_vencimiento']

for col in columnas_fecha:
    df_atb[col] = pd.to_datetime(df_atb[col], dayfirst=True, errors='coerce')
    df_ins[col] = pd.to_datetime(df_ins[col], dayfirst=True, errors='coerce')

# para comprobar que las columnas de fechas se convirtieron correctamente, podemos imprimir el tipo de dato de cada columna nuevamente:
print("df_atb.dtypes:", df_atb.dtypes)
print("df_ins.dtypes:", df_ins.dtypes)


# Paso 1: Identificamos las aperturas y les asignamos el valor -1 (descuento)
df_atb['unidad_consumida'] = df_atb['tipo_registro'].apply(lambda x: 1 if x == 'Apertura' else 0).astype('Int64')  
df_ins['unidad_consumida'] = df_ins['tipo_registro'].apply(lambda x: 1 if x == 'Apertura' else 0).astype('Int64')

print("df_atb", df_atb.tail(30))
print("df_ins", df_ins.tail(30))



df_atb.head():       marca_temporal tipo_registro                  antibiotico Responsable (iniciales) fecha_recepcion Cantidad (En caso de sensidiscos, indicar cantidad de tubos) fecha_apertura Fecha de cierre Motivo de cierre Comentario (opcional) fecha_vencimiento cantidad_eliminada
0  6/01/2026 8:02:24      Apertura       Aztreonam (Sensidisco)                     SCS                                                                                  29/08/2025                                                                                            
1  6/01/2026 8:02:39      Apertura       Amikacina (Sensidisco)                     SCS                                                                                  29/08/2025                                                                                            
2  6/01/2026 8:02:56      Apertura  Ciprofloxacino (Sensidisco)                     SCS                                                                                  

KeyError: 'cantidad_recibida'

#### Creación DataFrame Dashboard

In [ ]:

# Columna F: Consumo promedio en días por unidad (Lógica: Tiempo entre aperturas / Cantidad abierta) siempre que no haya eventos entre aperturas
# Columna G: Alertas de Vencimiento (Lógica: Si fecha_vencimiento - fecha_actual <= 30 días, entonces "Alerta", sino "OK")
# Columna H: Predicción de Quiebre de Stock (Lógica: Modelo de ML basado en consumo histórico y fechas de vencimiento)
# Columna I: Alertas de Consumo (Lógica: Si consumo promedio en días por unidad <= 7 días, entonces "Alerta", sino "OK")
# Columna J: Recomendación de Compra (Lógica: Si Stock Disponible <= 10 unidades o Predicción de Quiebre de Stock es "Alerta", entonces "Comprar", sino "No Comprar")
# Columna K: Historial de Consumo (Lógica: Gráfico de consumo histórico por antibiótico)
# Columna L: Historial de Eventos (Lógica: Lista de eventos registrados por antibiótico)
# Columna M: Historial de Vencimientos (Lógica: Lista de fechas de vencimiento por antibiótico)
# Columna N: Historial de Recepciones (Lógica: Lista de fechas y cantidades de recepciones por antibiótico)
# Columna O: Historial de Aperturas (Lógica: Lista de fechas de aperturas por antibiótico)
# Columna P: Historial de Eliminaciones (Lógica: Lista de fechas y cantidades de eliminaciones por antibiótico)
# para concatenar ambos dataframes, podemos usar la función pd.concat() de pandas, pero antes debemos asegurarnos de que ambos dataframes tengan las mismas columnas y el mismo formato, para luego aplicar las agregaciones necesarias para cada columna del dashboard.
df_atb = df_atb.rename(columns={'antibiotico': 'nombre_item'})
df_ins = df_ins.rename(columns={'insumo': 'nombre_item'})

df_atb['categoria'] = 'Antibiótico'
df_ins['categoria'] = 'Insumo'

df_atb_ins = pd.concat([df_atb, df_ins])

# df_dashboard.describe()
# print (df_atb_ins.head())
# print (df_atb_ins.tail())


# Asegurar que cantidad_eliminada sea numérica (Int64 para manejar <NA>)

df_atb_ins['cantidad_eliminada'] = pd.to_numeric(df_atb_ins['cantidad_eliminada'], errors='coerce').astype('Int64')

# Crear el dashboard consolidado usando pivot_table (más seguro que lambda)
df_pivot = df_atb_ins.pivot_table(
    index='nombre_item',
    columns='tipo_registro',
    values=['cantidad_recibida', 'unidad_consumida', 'cantidad_eliminada'],
    aggfunc='sum',
    fill_value=0 # Importante para no heredar valores de otros ítems [cite: 61]
).reset_index()

# Aplanar y limpiar nombres (evita columnas duplicadas como 'cantidad_eliminada_Apertura')
df_pivot.columns = ['_'.join(col).strip('_') for col in df_pivot.columns.values]

# 
df_dashboard = pd.DataFrame()
df_dashboard['nombre_item'] = df_pivot['nombre_item']
df_dashboard['Recibido'] = df_pivot.get('cantidad_recibida_Recepción', 0)
df_dashboard['Consumido'] = df_pivot.get('unidad_consumida_Apertura', 0)
df_dashboard['Descartes'] = (df_pivot.get('cantidad_eliminada_Cierre', 0)
)

# Columna E: Cálculo de Stock Disponible (Lógica: Recibido - Abierto - Eliminado)
df_dashboard['Stock actual'] = (
    df_dashboard['Recibido'].fillna(0) - 
    (df_dashboard['Consumido'].fillna(0) + df_dashboard['Descartes'].fillna(0))
)

# print(df_atb_ins.head(70)   )

# ---- CALC_LOGICA ----
# 1. Filtrar solo los eventos relevantes para el consumo y ordenar cronológicamente
df_calc_logica = df_atb_ins[df_atb_ins['tipo_registro'].isin(['Apertura', 'Evento'])].copy()
pd.set_option('display.max_rows', None)

#Para eliminar las columnas que no nos interesan para el calculo de la lógica:
columnas_a_eliminar = ['marca_temporal', 'fecha_recepcion', 'cantidad_recibida', 'fecha_vencimiento', 'Evento', 'cantidad_eliminada', 'unidad_consumida', 'categoría']
df_calc_logica = df_atb_ins.drop(columns=columnas_a_eliminar, errors='ignore')
df_calc_logica['fecha_calculo'] = np.where(
    df_calc_logica['tipo_registro'] == 'Apertura', 
    df_calc_logica['fecha_apertura'], 
    df_calc_logica['fecha_evento'])

df_calc_logica = df_calc_logica[df_calc_logica['tipo_registro'] != 'Recepción']
df_calc_logica = df_calc_logica.sort_values(by=['nombre_item', 'fecha_calculo'])

# 2. Identificar el tipo y la fecha del siguiente registro para comparar
df_calc_logica['sig_tipo'] = df_calc_logica.groupby('nombre_item')['tipo_registro'].shift(-1)
df_calc_logica['sig_fecha'] = df_calc_logica.groupby('nombre_item')['fecha_calculo'].shift(-1)

# 3. Calcular la diferencia de días entre eventos
df_calc_logica['dias_ciclo'] = (df_calc_logica['sig_fecha'] - df_calc_logica['fecha_calculo']).dt.days

# 4. Aplicar la validación: Solo Apertura -> Apertura es válido
# Si hay un 'Evento' (Descarte/Falla) a continuación, se ignora el ciclo
df_calc_logica['consumo_prom_dias'] = df_calc_logica.apply(
    lambda row: row['dias_ciclo'] if (row['tipo_registro'] == 'Apertura' and row['sig_tipo'] == 'Apertura') 
    else pd.NA, axis=1
  )

# 5. Convertir a Int64 para mantener la consistencia con celdas <NA>
df_calc_logica['consumo_prom_dias'] = df_calc_logica['consumo_prom_dias'].astype('Int64')

# print(df_calc_logica)

# Columna F: Consumo promedio en días por unidad (Lógica: Tiempo entre aperturas / Cantidad abierta) siempre que no haya eventos entre aperturas
# 1. Calculamos el promedio de días por cada producto
# Pandas ignorará automáticamente los valores <NA> o NaN en el cálculo del mean()
df_promedios = df_calc_logica.groupby('nombre_item')['consumo_prom_dias'].mean().reset_index()

# 2. Renombramos la columna para que sea clara en el Dashboard
df_promedios = df_promedios.rename(columns={'consumo_prom_dias': 'Consumo promedio'})

# 3. Integramos el cálculo al df_dashboard mediante un merge
df_dashboard = df_dashboard.merge(df_promedios, on='nombre_item', how='left')
df_dashboard['Consumo promedio'] = df_dashboard['Consumo promedio'].round(0).astype('Int64')
print(df_dashboard)

#Columna G: 

# 1. Calcular Días de Autonomía (Días de Stock Disponible)
# Lógica: stock_actual / consumo_promedio_diario
# Usamos un manejo de errores para evitar divisiones por cero o valores nulos
df_dashboard['dias_autonomia'] = (
    df_dashboard['Stock actual'] * df_dashboard['Consumo promedio']
).replace([float('inf'), -float('inf')], pd.NA)

# 2. Calcular la Fecha Estimada de Quiebre
# Lógica: Fecha de Hoy + Días de Autonomía
fecha_hoy = datetime.now()

def calcular_fecha_quiebre(dias):
    if pd.isna(dias) or dias < 0:
        return pd.NaT
    # Limitamos a un futuro razonable (ej. 10 años) para evitar errores de desbordamiento
    dias = min(dias, 3650) 
    return fecha_hoy + timedelta(days=float(dias))

df_dashboard['fecha_quiebre'] = df_dashboard['dias_autonomia'].apply(calcular_fecha_quiebre)

# 3. Formatear para visualización en Streamlit (Opcional)
df_dashboard['fecha_quiebre_display'] = df_dashboard['fecha_quiebre'].dt.strftime('%d/%m/%Y')

print(df_dashboard)

                                 nombre_item  Recibido  Consumido  Descartes  Stock actual  Consumo promedio
0                         Amikacina (E-test)         1          1          0             0              <NA>
1                     Amikacina (Sensidisco)        10          5          0             5                70
2   Amoxicilina/Ac. clavulánico (Sensidisco)         9          1          0             8              <NA>
3                    Ampicilina (Sensidisco)         5          2          0             3                83
4          Ampicilina/Sulbactam (Sensidisco)         4          1          0             3              <NA>
5                      Azitromicina (E-test)         1          1          0             0              <NA>
6                  Azitromicina (Sensidisco)         5          3          0             2               106
7                         Aztreonam (E-test)         2          1          0             1              <NA>
8                  

#### Análisis

In [ ]:
# Replicamos el IF: Si es Apertura usa G (Fecha_Apertura), si es Evento usa H (Fecha_Evento)
# #df_processed = df_atb.copy()

# Creamos la columna 'fecha' condicionalmente
#df_processed['fecha_calc_logica'] = df_processed.apply(
#    lambda row: row['fecha_apertura'] if row['tipo_registro'] == 'Apertura' 
#    else (row['fecha_evento'] if row['tipo_registro'] == 'Evento' else None), 
#    axis=1
#    
#)

# Seleccionamos y renombramos columnas para que coincidan con el encabezado final
# Usamos .index + 2 para replicar el ROW() de Sheets
# calc_logica = pd.DataFrame({
#    'fecha': df_processed['fecha_calc_logica'],
#    'tipo_evento': df_processed['tipo_registro'],
#    'antibiotico': df_processed['antibiotico'],
#    'ID_Original': df_processed.index + 2
#})

# --- PROCESAMIENTO DE INSUMOS ---
# df_ins_final = pd.DataFrame({
#     'fecha': df_ins['fecha_apertura'],
#    'tipo_evento': df_ins['tipo_registro'],
#     'insumo': df_ins['insumo'],
#    'ID_Original': df_ins.index + 2
# })

# --- COMBINACIÓN (El { ... ; ... } de la fórmula) ---
# Concatenamos ambos DataFrames verticalmente


# --- QUERY (SELECT, WHERE, ORDER BY) ---

#para comprobar el tipo de dato de la columna fecha_calc_logica, podemos usar el siguiente código:
#print(calc_logica.dtypes)

# calc_logica['fecha_calc_logica'] = pd.to_datetime(calc_logica['fecha_calc_logica'])

# 2. Filtrar: Donde Col2 sea 'Apertura' o 'Evento'
# 3. Ordenar: Por Col3 (insumo) y luego Col1 (fecha)
#calc_logica = calc_logica[
#    calc_logica['tipo_evento'].isin(['Apertura', 'Evento'])
#].sort_values(by=['insumo', 'fecha'])

# Limpiar filas donde la fecha haya quedado vacía (opcional, como el WHERE Col1 <> "")
#calc_logica = calc_logica.dropna(subset=['fecha'])

# --------------------------------------------------------------------------


# 1. Calcular el promedio de días en la tabla Calc_Logica
# Agrupamos por 'insumo' y calculamos la media de la columna 'Días'
# Pandas ignora automáticamente los valores nulos (NaN) al calcular el promedio
#promedios = calc_logica.groupby('antibiotico')['fecha'].mean().reset_index()

# Renombramos la columna para que sea clara
#promedios = promedios.rename(columns={'fecha': 'consumo_promedio'})

# 2. Unir los promedios a la tabla Dashboard_Stock
# Supongamos que df_dashboard es tu tabla principal con la columna 'Producto'
#df_dashboard = pd.merge(
#    df_dashboard, 
#    promedios, 
#    left_on='Producto', 
#    right_on='antibiotico', 
#    how='left'
#)

# 3. Replicar el IFERROR( ... ; "Sin datos")
# Reemplazamos los valores nulos (donde no hubo coincidencias o datos) por el texto
# df_dashboard['consumo_promedio'] = df_dashboard['consumo_promedio'].fillna('Sin datos')

# Opcional: Si prefieres que los números se vean con pocos decimales antes de convertir a texto
# df_dashboard['consumo_promedio'] = df_dashboard['consumo_promedio'].apply(
#     lambda x: round(x, 2) if isinstance(x, (int, float)) else x
# )

# print(df_dashboard[['Producto', 'consumo_promedio']])